In [9]:
import pandas as pd
import re

# Open the Excel file
df = pd.read_excel('../../data/comments/comments.xlsx')

# Drop rows where the Comments column is empty
df = df.dropna(subset=['Comments'])

# Save the changes to the Excel file
df.to_excel('../../data/comments/comments_stage1.xlsx', index=False)

In [10]:
df = pd.read_excel('../../data/comments/comments_stage1.xlsx')

# Check if there are any empty rows under the Comments column
empty_rows = df['Comments'].isna().sum()

# Print the number of empty rows
print(f"Number of empty rows under Comments column: {empty_rows}")


Number of empty rows under Comments column: 0


In [17]:
import pandas as pd
import re

def remove_emojis(text):
    """Remove emojis from a given text."""
    emoji_pattern = re.compile(
        "[" 
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F700-\U0001F77F"  # alchemical symbols
        u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U0001FA00-\U0001FA6F"  # Chess Symbols
        u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        u"\U00002700-\U000027BF"  # Dingbats
        u"\U00002600-\U000026FF"  # Miscellaneous Symbols
        u"\U00002000-\U0000209F"  # General Punctuation
        u"\U00002300-\U000023FF"  # Miscellaneous Technical
        u"\U00002B50-\U00002BFF"  # Miscellaneous Symbols and Arrows
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def clean_and_save_data(input_file, output_file, comments_column='Comments'):
    """Clean the 'Comments' column by removing emojis and save to an Excel file."""
    # Load the Excel file into a pandas DataFrame
    df = pd.read_excel(input_file, engine="openpyxl")
    
    # Check if 'Comments' column exists
    if comments_column in df.columns:
        # Convert all entries in the 'Comments' column to strings and remove emojis
        df[comments_column] = df[comments_column].astype(str).apply(remove_emojis)
        
        # Remove rows where 'Comments' column is empty or contains only whitespace
        df = df[df[comments_column].str.strip().astype(bool)]
        
        # Save the cleaned DataFrame to a new Excel file
        df.to_excel(output_file, index=False)
        print(f"Emojis removed and blank rows dropped. Cleaned data saved to '{output_file}'")
    else:
        print(f"Error: '{comments_column}' column not found in the input file.")

# Specify input and output files
input_file = "../../data/comments/comments_stage1.xlsx"  # Replace with your input Excel file path
output_file = "cleaned_comments.xlsx"  # Replace with your desired output Excel file path

# Run the cleaning and saving function
clean_and_save_data(input_file, output_file)


Emojis removed and blank rows dropped. Cleaned data saved to 'cleaned_comments.xlsx'


In [12]:
df = pd.read_excel('cleaned_comments.xlsx')

# Check if there are any empty rows under the Comments column
empty_rows = df['Comments'].isna().sum()

# Print the number of empty rows
print(f"Number of empty rows under Comments column: {empty_rows}")

Number of empty rows under Comments column: 0


In [13]:
import pandas as pd
import nltk
from nltk.corpus import words

# Load the cleaned comments from the Excel file
df = pd.read_excel('cleaned_comments.xlsx')

# Tokenize the comments into words
nltk.download('words')
english_words = set(words.words())

def remove_english_words(text):
    """Remove English words from a given text."""
    tokens = text.split()
    filtered_tokens = [token for token in tokens if token.lower() not in english_words]
    return ' '.join(filtered_tokens)

# Apply the remove_english_words function to the 'Comments' column
df['Comments'] = df['Comments'].apply(remove_english_words)

# Save the updated DataFrame to the Excel file
df.to_excel('cleaned_comments_no_english.xlsx', index=False)

[nltk_data] Downloading package words to C:\Users\Zenith
[nltk_data]     Anthony\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


In [20]:
import pandas as pd
import pandas as pd
import re

def extract_english_comments(input_file, output_file):
    # Load the DataFrame from the existing Excel file
    df = pd.read_excel(input_file)

    # Define a regex pattern to match rows with English letters
    pattern = re.compile(r'[a-zA-Z]')

    # Filter the rows with English letters
    english_df = df[df['Comments'].str.contains(pattern, na=False)]

    # Open the output file in write mode
    with open(output_file, 'w', encoding='utf-8') as f:
        # Iterate over the filtered DataFrame and write each comment to the file
        for comment in english_df['Comments']:
            # Remove newlines within comments, add a comma at the end, and write to the file
            cleaned_comment = comment.replace('\n', ' ') + ','
            f.write(cleaned_comment + '\n')

    print(f"English comments extracted and saved to '{output_file}'")

# Input and output file paths
input_file = 'cleaned_comments.xlsx'
output_file = 'english_comments.txt'

# Extract English comments and save to a text file
extract_english_comments(input_file, output_file)


English comments extracted and saved to 'english_comments.txt'


In [22]:
import pandas as pd
import nltk
from transformers import pipeline

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

# Load the comments from the Excel file
input_file = 'cleaned_comments.xlsx'
output_file = 'labeled_comments.xlsx'
column_name = 'Comments'

try:
    df = pd.read_excel(input_file, engine='openpyxl')
except Exception as e:
    print(f"Error reading the input file: {e}")
    raise

# Initialize the text classification pipeline with a pre-trained model
classifier = pipeline('text-classification', model='nlptown/bert-base-multilingual-uncased-sentiment')

# Define a function to classify comments
def classify_comment(comment):
    try:
        result = classifier(comment)[0]
        label = result['label']
        if '1 star' in label:
            return 'opinion'
        elif '5 stars' in label:
            return 'claim'
        else:
            return 'unknown'  # default to unknown if uncertain
    except Exception as e:
        print(f"Error classifying comment '{comment}': {e}")
        return 'unknown'

# Apply classification to the comments
if column_name in df.columns:
    df['Label'] = df[column_name].apply(lambda x: classify_comment(str(x)))

    # Save the labeled DataFrame back to a new Excel file
    try:
        df.to_excel(output_file, index=False, engine='openpyxl')
        print(f"Labeled comments saved to '{output_file}'")
    except Exception as e:
        print(f"Error saving the output file: {e}")
else:
    print(f"Error: Column '{column_name}' not found in the input file.")


[nltk_data] Downloading package punkt to C:\Users\Zenith
[nltk_data]     Anthony\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Zenith
[nltk_data]     Anthony\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Labeled comments saved to 'labeled_comments.xlsx'


In [25]:
# Path to the labeled Excel file
labeled_file = 'labeled_comments.xlsx'

try:
    # Read the labeled Excel file
    labeled_df = pd.read_excel(labeled_file, engine='openpyxl')

    # Count the number of 'unknown' labels
    unknown_count = labeled_df['Label'].value_counts().get('unknown', 0)
    print(f"Number of comments labeled as 'unknown': {unknown_count}")
except Exception as e:
    print(f"Error reading the labeled output file: {e}")

Number of comments labeled as 'unknown': 12159
